In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count, sum as spark_sum

spark = SparkSession.builder.appName("TugasMandiri4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil dibuat!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/13 13:37:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/13 13:38:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession berhasil dibuat!


In [2]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [3]:
df_tugas = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv", header=True, inferSchema=True)

df_tugas.printSchema()
print("Jumlah total baris:", df_tugas.count())
df_tugas.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah total baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|  

In [5]:
# Menghitung jumlah nilai kosong (null/NaN) pada kolom rating
null_count = df_tugas.filter(col("rating").isNull()).count()
print("Jumlah data kosong pada kolom rating:", null_count)

df_clean = df_tugas.na.fill({"rating": 0})

# Menampilkan 5 baris pertama dari data yang sudah dibersihkan
df_clean.show(5)

Jumlah data kosong pada kolom rating: 204
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
+--------+-------------------+----------------

Metode `df.na.fill({"rating": 0})` dipilih untuk menangani nilai kosong pada kolom `rating` agar baris data transaksi tersebut tidak terbuang. Menggunakan metode `df.na.drop()` akan menghapus seluruh baris transaksi yang nilai rating-nya kosong, sehingga informasi transaksi penting lainnya seperti `unit_terjual`, `harga_satuan`, dan pendapatan akan ikut hilang. Pengisian nilai `0` menjadi solusi terbaik untuk menandai transaksi yang tidak atau belum diberikan rating oleh pembeli tanpa membuang data finansialnya.

In [6]:
# Menambahkan kolom total_pendapatan dan tier_transaksi
df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
                         .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

# Menampilkan 10 baris pertama
df_transformed.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [8]:
# 1. Kategori dengan total pendapatan tertinggi
print("--- 1. Kategori Total Pendapatan Tertinggi ---")
df_transformed.groupBy("kategori").agg(spark_sum("total_pendapatan").alias("total_pendapatan")).orderBy(col("total_pendapatan").desc()).show(1)

# 2. Kota dengan jumlah transaksi tier "Besar" terbanyak
print("--- 2. Kota Transaksi Tier Besar Terbanyak ---")
df_transformed.filter(col("tier_transaksi") == "Besar").groupBy("kota").count().orderBy(col("count").desc()).show(1)

# 3. Rata-rata rating untuk masing-masing metode pembayaran
print("--- 3. Rata-rata Rating per Metode Pembayaran ---")
df_transformed.groupBy("metode_pembayaran").agg(avg("rating").alias("rata_rata_rating")).orderBy(col("rata_rata_rating").desc()).show()

--- 1. Kategori Total Pendapatan Tertinggi ---
+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row

--- 2. Kota Transaksi Tier Besar Terbanyak ---
+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+
only showing top 1 row

--- 3. Rata-rata Rating per Metode Pembayaran ---
+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|         E-Wallet|             3.292|
|     Kartu Kredit|3.1910569105691056|
+-----------------+------------------+



In [9]:
# Menyimpan DataFrame hasil bagian C ke HDFS
df_transformed.write.mode("overwrite").csv("hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olah_september.csv", header=True)
print("Berhasil menyimpan data ke HDFS.")

# Verifikasi membaca kembali data dari HDFS
df_verifikasi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olah_september.csv", header=True, inferSchema=True)
df_verifikasi.show(5)

Berhasil menyimpan data ke HDFS.
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|   

### Penjelasan
Spark menyimpan output dalam bentuk folder berisi beberapa berkas partisi (`part-00000...`) karena sifat arsitekturnya yang terdistribusi (*distributed processing*). Data dipecah ke dalam beberapa partisi di memori (worker node) dan diproses secara sejajar (*parallel*). Saat proses penulisan ke HDFS, setiap worker node menulis bagian partisinya sendiri secara independen untuk memaksimalkan performa I/O daripada mengantre membuat satu berkas tunggal.